# Milestone 1

This milestone focuses on understanding the dataset and establishing a baseline performance through **exploratory data analysis (EDA)** and simple **heuristic-based methods** using `librosa`.

---

## Suggested Readings
- [Hugging Face Audio Course](https://huggingface.co/learn/audio-course/en/chapter0/introduction)
- [Librosa Documentation](https://librosa.org/doc/main/core.html#audio-loading)

---

## Instructions
Use this notebook to answer **all Milestone-1 questions**.

---

## Resources
- Notebook Link:  
  https://colab.research.google.com/drive/1m6UczhxQIke_raWSqukSWuiKbIVt7MMb?usp=sharing  

- Competition Link:  
  https://www.kaggle.com/competitions/jan-2026-dl-gen-ai-project/


In [1]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch

import warnings
warnings.filterwarnings("ignore")

In [2]:
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [25]:
# CONFIGURATION
DATA_ROOT = '../data/messy_mashup/genres_stems' # Enter dataset path
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock'] # Make the list of all genres available (alphabetical order)
STEMS = ['bass.wav', 'drums.wav', 'other.wav', 'vocals.wav'] # Write here stems file name (dataset uses "other.wav", not "others.wav")
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
GENRE_TO_TEST = 'rock'
SONG_INDEX = 0  # First song from rock genre #Enter index as per Q10. 

In [26]:
# Verifying dataset structure
print("Dataset Structure Verification")
for g in GENRES:
    gpath = os.path.join(DATA_ROOT, g)
    if os.path.isdir(gpath):
        songs = sorted([d for d in os.listdir(gpath) if os.path.isdir(os.path.join(gpath, d))])
        print(f"{g}: {len(songs)} songs")
        if songs:
            # Show stems of first song
            first_song = os.path.join(gpath, songs[0])
            files = os.listdir(first_song)
            print(f"  First song stems: {sorted(files)}")
    else:
        print(f"{g}: FOLDER NOT FOUND")

Dataset Structure Verification
blues: 100 songs
  First song stems: ['bass.wav', 'drums.wav', 'other.wav', 'vocals.wav']
classical: 100 songs
  First song stems: ['bass.wav', 'drums.wav', 'other.wav', 'vocals.wav']
country: 100 songs
  First song stems: ['bass.wav', 'drums.wav', 'other.wav', 'vocals.wav']
disco: 100 songs
  First song stems: ['bass.wav', 'drums.wav', 'other.wav', 'vocals.wav']
hiphop: 100 songs
  First song stems: ['bass.wav', 'drums.wav', 'other.wav', 'vocals.wav']
jazz: 100 songs
  First song stems: ['bass.wav', 'drums.wav', 'other.wav', 'vocals.wav']
metal: 100 songs
  First song stems: ['bass.wav', 'drums.wav', 'other.wav', 'vocals.wav']
pop: 100 songs
  First song stems: ['bass.wav', 'drums.wav', 'other.wav', 'vocals.wav']
reggae: 100 songs
  First song stems: ['bass.wav', 'drums.wav', 'other.wav', 'vocals.wav']
rock: 100 songs
  First song stems: ['bass.wav', 'drums.wav', 'other.wav', 'vocals.wav']


In [27]:
def build_dataset(root_dir, val_split=0.17, seed=42):
    # Initialize empty dictionaries
    # Determine actual stem filenames from the dataset
    train_dataset = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    val_dataset   = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    rng = random.Random(seed)

    skipped_incomplete = 0
    skipped_corrupted = 0
    all_file_sizes = []  # To answer Q1 and Q2

    for genre in GENRES:
        genre_path = os.path.join(root_dir, genre)
        if not os.path.isdir(genre_path):
            print(f"WARNING: Genre folder not found: {genre_path}")
            continue

        songs = sorted([d for d in os.listdir(genre_path)
                       if os.path.isdir(os.path.join(genre_path, d))])
        valid_songs = []

        for song in songs:
            song_path = os.path.join(genre_path, song)

            # CHECK: Completeness (all stems present?)
            all_present = True
            for stem in STEMS:
                stem_path = os.path.join(song_path, stem)
                if not os.path.exists(stem_path):
                    all_present = False
                    break
            if not all_present:
                skipped_incomplete += 1
                continue

            # CHECK: Corruption (any file < 4KB?)
            corrupted = False
            for stem in STEMS:
                stem_path = os.path.join(song_path, stem)
                size = os.path.getsize(stem_path)
                all_file_sizes.append({
                    'genre': genre,
                    'song': song,
                    'stem': stem,
                    'path': stem_path,
                    'size_bytes': size
                })
                if size < 4 * 1024:  # 4KB = 4 * 1024 bytes
                    corrupted = True

            if corrupted:
                skipped_corrupted += 1
                continue

            valid_songs.append(song_path)

        # Stratified Shuffle Split
        rng.shuffle(valid_songs)
        n_val = int(len(valid_songs) * val_split)
        val_songs = valid_songs[:n_val]
        train_songs = valid_songs[n_val:]

        def add_to_dict(target_dict, song_list):
            for sp in song_list:
                for stem in STEMS:
                    stem_key = stem.replace('.wav', '')
                    target_dict[genre][stem_key].append(os.path.join(sp, stem))

        add_to_dict(train_dataset, train_songs)
        add_to_dict(val_dataset, val_songs)

    print(f"\nSkipped (incomplete stems): {skipped_incomplete}")
    print(f"Skipped (corrupted <4KB): {skipped_corrupted}")

    return train_dataset, val_dataset, all_file_sizes

tr, val, all_file_sizes = build_dataset(DATA_ROOT)


Skipped (incomplete stems): 0
Skipped (corrupted <4KB): 0


In [28]:
# ANSWER Q1: corrupted (<4KB) + sounds < 5.0491MB

df_sizes = pd.DataFrame(all_file_sizes)

threshold_4kb = 4 * 1024
threshold_low = 5.0491 * 1024 * 1024  # 5.0491 MB in bytes

n_corrupted = len(df_sizes[df_sizes['size_bytes'] < threshold_4kb])
n_below_5_0491 = len(df_sizes[df_sizes['size_bytes'] < threshold_low])

print(f"\nQ1\n")
print(f"Corrupted files (<4KB): {n_corrupted}")
print(f"Files < 5.0491MB: {n_below_5_0491}")
print(f"Answer Q1 = {n_corrupted + n_below_5_0491}")


# ANSWER Q2: |sounds > 5.0493MB - sounds < 5.0491MB|

threshold_high = 5.0493 * 1024 * 1024

n_above_5_0493 = len(df_sizes[df_sizes['size_bytes'] > threshold_high])

print(f"\nQ2\n")
print(f"Files > 5.0493MB: {n_above_5_0493}")
print(f"Files < 5.0491MB: {n_below_5_0491}")
print(f"Answer Q2 = |{n_above_5_0493} - {n_below_5_0491}| = {abs(n_above_5_0493 - n_below_5_0491)}")


# ANSWER Q3: |training reggae drums - validation country vocals|

n_train_reggae_drums = len(tr['reggae']['drums'])
n_val_country_vocals = len(val['country']['vocals'])

print(f"\nQ3\n")
print(f"Training reggae drums: {n_train_reggae_drums}")
print(f"Validation country vocals: {n_val_country_vocals}")
print(f"Answer Q3 = |{n_train_reggae_drums} - {n_val_country_vocals}| = {abs(n_train_reggae_drums - n_val_country_vocals)}")



Q1

Corrupted files (<4KB): 0
Files < 5.0491MB: 1256
Answer Q1 = 1256

Q2

Files > 5.0493MB: 184
Files < 5.0491MB: 1256
Answer Q2 = |184 - 1256| = 1072

Q3

Training reggae drums: 83
Validation country vocals: 17
Answer Q3 = |83 - 17| = 66


In [29]:
def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
    """
    Input:
        dataset_dict: The dictionary structure {genre: {stem: [paths...]}}
    Output:
        df: Pandas DataFrame containing details of all files with silence >= 5s
    """

    records = []
    total_files = sum(
        len(paths) for g in dataset_dict for s, paths in dataset_dict[g].items()
    )
    print(f"Scanning {total_files} files for long silences...")

    for genre in dataset_dict:
        for stem_name in dataset_dict[genre]:
            for file_path in tqdm(
                dataset_dict[genre][stem_name],
                desc=f"{genre}/{stem_name}",
                leave=False
            ):
                try:
                    y, _ = librosa.load(file_path, sr=sr)
                except Exception as e:
                    print(f"Error loading {file_path}: {e}")
                    continue

                total_duration = len(y) / sr
                intervals = librosa.effects.split(y, top_db=top_db)

                silence_gaps = []  # list of (duration_sec, location_type)

                if len(intervals) == 0:
                    # CASE A: Fully silent file
                    silence_gaps.append((total_duration, "full"))
                else:
                    # CASE B: START silence (before first non-silent interval)
                    start_silence = intervals[0][0] / sr
                    if start_silence > 0:
                        silence_gaps.append((start_silence, "start"))

                    # CASE D: MIDDLE silences (gaps between non-silent intervals)
                    for i in range(1, len(intervals)):
                        gap = (intervals[i][0] - intervals[i - 1][1]) / sr
                        if gap > 0:
                            silence_gaps.append((gap, "middle"))

                    # CASE C: END silence (after last non-silent interval)
                    end_silence = (len(y) - intervals[-1][1]) / sr
                    if end_silence > 0:
                        silence_gaps.append((end_silence, "end"))

                # Find maximum silence duration
                if silence_gaps:
                    max_silence = max(g[0] for g in silence_gaps)
                else:
                    max_silence = 0.0

                # Collect all silence location types where that specific gap >= threshold
                silence_type = []
                for gap_dur, gap_loc in silence_gaps:
                    if gap_dur >= threshold_sec:
                        if gap_loc not in silence_type:
                            silence_type.append(gap_loc)

                if max_silence >= threshold_sec:
                    records.append({
                        "Genre": genre,
                        "Stem": stem_name,
                        "Duration": round(total_duration, 2),
                        "Max_Silence_Sec": round(max_silence, 2),
                        "Silence_Location": ", ".join(silence_type),
                        "File_Path": file_path
                    })

    df = pd.DataFrame(records)
    return df


print("\nRunning silence detection\n")
df_silence = find_long_silences(tr, threshold_sec=DURATION, top_db=TOP_DB)
print("\nSilence detection run completed.\n")


Running silence detection

Scanning 3320 files for long silences...



Silence detection run completed.



In [30]:
# Pivot table for reference
if len(df_silence) > 0:
    pivot = pd.pivot_table(df_silence, index='Genre', columns='Stem',
                           values='File_Path', aggfunc='count', fill_value=0)
    print("\nPivot Table: Count by Genre vs Stem\n")
    print(pivot)


Pivot Table: Count by Genre vs Stem

Stem       bass  drums  other  vocals
Genre                                
blues        17     29      4      44
classical    68     58      5      69
country      12     14      1      16
disco         8      2      2      17
hiphop       22      2     21       6
jazz         24     22      1      73
metal         7      2      1      41
pop          11      6      2       5
reggae        5      5      7      15
rock         12      5      1      26


In [31]:

# ANSWER Q4: Total files with silence >= 5s

print(f"\nQ4\n")
print(f"Total files with silence >= 5s: {len(df_silence)}")

# ANSWER Q5: Total vocals tracks with silence >= 5s

vocals_silence = df_silence[df_silence['Stem'] == 'vocals']
print(f"\n Q5 \n")
print(f"Vocals tracks with silence >= 5s: {len(vocals_silence)}")

# ANSWER Q6: Average silence length in Vocals

avg_vocal_silence = round(vocals_silence['Max_Silence_Sec'].mean(), 2)
print(f"\n Q6 \n")
print(f"Average silence length in Vocals: {avg_vocal_silence} secs")


# ANSWER Q7: Drums in jazz with silence >= 5s

jazz_drums = df_silence[(df_silence['Genre'] == 'jazz') & (df_silence['Stem'] == 'drums')]
print(f"\n Q7 \n")
print(f"Jazz drums with silence >= 5s: {len(jazz_drums)}")


# ANSWER Q8: Jazz drums, silence >= 5s, location is ONLY middle

jazz_drums_middle_only = jazz_drums[jazz_drums['Silence_Location'] == 'middle']
print(f"\n Q8 \n")
print(f"Jazz drums, silence >= 5s, only middle: {len(jazz_drums_middle_only)}")


# ANSWER Q9: Jazz drums, silence >= 5s, Max_Silence_Sec >= 10

jazz_drums_10 = jazz_drums[jazz_drums['Max_Silence_Sec'] >= 10]
print(f"\n Q9 \n")
print(f"Jazz drums, silence >= 5s and Max_Silence >= 10s: {len(jazz_drums_10)}")


Q4

Total files with silence >= 5s: 688

 Q5 

Vocals tracks with silence >= 5s: 312

 Q6 

Average silence length in Vocals: 12.54 secs

 Q7 

Jazz drums with silence >= 5s: 22

 Q8 

Jazz drums, silence >= 5s, only middle: 12

 Q9 

Jazz drums, silence >= 5s and Max_Silence >= 10s: 8


In [32]:
stems_audio = []
try:
    for key in STEM_KEYS:

        # STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
        # Get the first song's stem path
        file_path = tr[GENRE_TO_TEST][key][SONG_INDEX]
        y, _ = librosa.load(file_path, sr=SR, duration=DURATION)
        stems_audio.append(y)
        print(f"  Loaded {key}: {len(y)} samples from {file_path}")

    print("Audio loaded successfully.")

    print("Audio loaded successfully.")
except NameError:
    print("ERROR: 'tr' dictionary not found. Please run build_dataset() first.")
except IndexError:
    print(f"ERROR: Song index {SONG_INDEX} out of range for genre {GENRE_TO_TEST}.")
except Exception as e:
    print(f"ERROR: {e}")

  Loaded drums: 110250 samples from ../data/messy_mashup/genres_stems\rock\rock.00082\drums.wav
  Loaded vocals: 110250 samples from ../data/messy_mashup/genres_stems\rock\rock.00082\vocals.wav
  Loaded bass: 110250 samples from ../data/messy_mashup/genres_stems\rock\rock.00082\bass.wav
  Loaded other: 110250 samples from ../data/messy_mashup/genres_stems\rock\rock.00082\other.wav
Audio loaded successfully.
Audio loaded successfully.


In [34]:
# ------------------- write your code here -------------------------------
# Stack them into a numpy array (Shape: 4 x Samples)
stems_stack = np.stack(stems_audio)
print(f"\nStems stack shape: {stems_stack.shape}")

# Mix the stems by summing them element-wise
mix_raw = np.sum(stems_stack, axis=0)
print(f"Mix shape: {mix_raw.shape}")
print(f"Length of mix sample: {len(mix_raw)}")

# Calculate RMS Amplitude MANUALLY
rms_val = np.sqrt(np.mean(mix_raw ** 2))
print(f"RMS Amplitude of mix: {round(rms_val, 2)}")

#Peak Normalization
max_val = np.max(np.abs(mix_raw))
print(f"Max absolute value (before normalization): {max_val}")


if max_val > 0:
    mix_norm = mix_raw / max_val
else:
    mix_norm = mix_raw

print(f"Max value of peak normalized sample: {round(np.max(mix_norm), 2)}")
# VALIDATION
assert np.isclose(np.max(np.abs(mix_norm)), 1.0), "Normalization failed."
#------------------------------------------------------------------------



Stems stack shape: (4, 110250)
Mix shape: (110250,)
Length of mix sample: 110250
RMS Amplitude of mix: 0.09000000357627869
Max absolute value (before normalization): 0.5610572099685669
Max value of peak normalized sample: 1.0
